# Project 3: Sports Analytics
Evaluate player performance, match results, and win probabilities to optimize tactical team strategies.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

df_matches = pd.read_csv("datasets/sports_matches_data.csv")
df_players = pd.read_csv("datasets/sports_players_data.csv")
print("Matches Data Shape:", df_matches.shape)
print("Players Data Shape:", df_players.shape)

Matches Data Shape: (2000, 10)
Players Data Shape: (5000, 11)


## 1. Match Performance & Win Distributions
Analyze match outcomes and overall statistics. Note that the Thunderbolts have a general 65% win probability.

In [2]:
# Check Team wins
wins_team_a = df_matches[df_matches['Outcome'] == 'Team A Win']['TeamA'].value_counts()
wins_team_b = df_matches[df_matches['Outcome'] == 'Team B Win']['TeamB'].value_counts()

total_wins = wins_team_a.add(wins_team_b, fill_value=0).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=total_wins.values, y=total_wins.index, hue=total_wins.index, palette='crest', legend=False)
plt.title("Total Wins by Team in 2025")
plt.xlabel("Number of Wins")
plt.tight_layout()
plt.savefig("visualizations/team_wins.png")
plt.savefig("../visualizations/team_wins.png")
plt.show()

print("Wins breakdown:")
print(total_wins)

Wins breakdown:
TeamA
Thunderbolts    241
Knights         210
Vipers          181
Gladiators      180
Wizards         180
Centurions      163
Ironclads       163
Strikers        163
Phoenix FC      161
Titans          154
Name: count, dtype: int64


C:\Users\HARISH RAGHAVENDER\AppData\Local\Temp\ipykernel_23292\2169587291.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Player Performance Analysis
We look at player rating, goals, and assists by position.

In [3]:
# Correlation Heatmap of Player Stats
corr = df_players[['Goals', 'Assists', 'Tackles', 'PassAccuracy', 'MinutesPlayed', 'Rating']].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Player Statistics Correlation Heatmap")
plt.tight_layout()
plt.savefig("visualizations/player_heatmap.png")
plt.savefig("../visualizations/player_heatmap.png")
plt.show()

# Metric summary by position
pos_metrics = df_players.groupby("Position")[["Goals", "Assists", "Tackles", "PassAccuracy", "Rating"]].mean().reset_index()
print(pos_metrics)

plt.figure(figsize=(10, 6))
df_melt = pd.melt(df_players, id_vars=['Position'], value_vars=['Goals', 'Assists'], var_name='Metric', value_name='Value')
sns.barplot(data=df_melt, x='Position', y='Value', hue='Metric', palette='muted')
plt.title("Average Goals and Assists by Position")
plt.tight_layout()
plt.savefig("visualizations/position_metrics.png")
plt.savefig("../visualizations/position_metrics.png")
plt.show()

C:\Users\HARISH RAGHAVENDER\AppData\Local\Temp\ipykernel_23292\1939211771.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


     Position     Goals   Assists   Tackles  PassAccuracy    Rating
0    Defender  0.047694  0.076622  5.938233     79.755121  7.196794
1     Forward  0.559727  0.278157  0.994881     80.248635  7.394369
2  Goalkeeper  0.000000  0.000000  0.000000     62.330365  6.526142
3  Midfielder  0.177069  0.521027  3.019920     79.829703  7.115095


C:\Users\HARISH RAGHAVENDER\AppData\Local\Temp\ipykernel_23292\1939211771.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Machine Learning Model: Match Outcome Prediction
Train a model to predict match outcomes based on team possession, shots, and historical win probabilities.

In [4]:
# Encode outcome as Team A Win (1), Team B Win (2), Draw (0)
outcome_map = {'Team A Win': 1, 'Team B Win': 2, 'Draw': 0}
df_matches['OutcomeLabel'] = df_matches['Outcome'].map(outcome_map)

X = df_matches[['Possession_TeamA', 'Shots_TeamA', 'WinProbability']]
y = df_matches['OutcomeLabel']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("Classification Report for Match Outcomes:")
print(classification_report(y_test, y_pred, target_names=['Draw', 'Team A Win', 'Team B Win']))

# Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Draw', 'Team A Win', 'Team B Win'])

plt.figure(figsize=(8, 8))
disp.plot(cmap='Blues')
plt.title("Match Outcome Prediction Confusion Matrix")
plt.tight_layout()
plt.savefig("visualizations/match_prediction_confusion.png")
plt.savefig("../visualizations/match_prediction_confusion.png")
plt.show()

Classification Report for Match Outcomes:
              precision    recall  f1-score   support

        Draw       0.04      0.02      0.02        61
  Team A Win       0.52      0.55      0.54       301
  Team B Win       0.41      0.44      0.42       238

    accuracy                           0.45       600
   macro avg       0.32      0.34      0.33       600
weighted avg       0.43      0.45      0.44       600



C:\Users\HARISH RAGHAVENDER\AppData\Local\Temp\ipykernel_23292\55143826.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
